In [ ]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [3]:
spark = SparkSession.builder \
    .appName("Week5Assignment") \
    .getOrCreate()

In [4]:
# Upload Dataset
from google.colab import files
uploaded = files.upload()

Saving sales_dataset.csv to sales_dataset.csv


In [6]:
df = spark.read.csv(
    "sales_dataset.csv",
    header=True,
    inferSchema=True
)

df.show(10)
df.printSchema()

+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|user_id|transaction_date|region|product_category|sale_amount| city|age|subscription|price|store_id|             email|username|   status|   raw_timestamp|
+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|   1001|      01-01-2025|  West|     Electronics|        100|Delhi| 18|     Premium| NULL|    S101|              NULL|    NULL|     NULL|01-01-2025 10:00|
|   1002|      02-01-2025|  West|       Furniture|        105|Delhi| 19|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|
|   1003|      03-01-2025|  West|        Clothing|        110|Delhi| 20|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|03-01-2025 10:02|
|   1004|      04-01-2025|  West|         Grocery|        115|De

Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a
specific set of columns: user_id and transaction_date.

In [7]:
df = df.dropDuplicates(["user_id", "transaction_date"])

df.show()

+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|user_id|transaction_date|region|product_category|sale_amount| city|age|subscription|price|store_id|             email|username|   status|   raw_timestamp|
+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|   1001|      01-01-2025|  West|     Electronics|        100|Delhi| 18|     Premium| NULL|    S101|              NULL|    NULL|     NULL|01-01-2025 10:00|
|   1002|      02-01-2025|  West|       Furniture|        105|Delhi| 19|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|
|   1003|      03-01-2025|  West|        Clothing|        110|Delhi| 20|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|03-01-2025 10:02|
|   1004|      04-01-2025|  West|         Grocery|        115|De

Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is
'West' and then group by product_category to find the average sale_amount.

In [8]:
from pyspark.sql.functions import col, avg

df.filter(col("region") == "West") \
  .groupBy("product_category") \
  .agg(avg("sale_amount").alias("Average_Sale")) \
  .show()

+----------------+------------+
|product_category|Average_Sale|
+----------------+------------+
|         Grocery|       405.0|
|     Electronics|       390.0|
|        Clothing|       400.0|
|       Furniture|       395.0|
+----------------+------------+



Q5. What is the difference between .na.drop() and .na.fill()? Provide a code example of
filling null values in a status column with the string 'Unknown'

In [9]:
df.na.drop().show()

+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|user_id|transaction_date|region|product_category|sale_amount| city|age|subscription|price|store_id|             email|username|   status|   raw_timestamp|
+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|   1002|      02-01-2025|  West|       Furniture|        105|Delhi| 19|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|
|   1003|      03-01-2025|  West|        Clothing|        110|Delhi| 20|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|03-01-2025 10:02|
|   1004|      04-01-2025|  West|         Grocery|        115|Delhi| 21|       Basic|  115|    S101|user1004@gmail.com|user1004|Completed|04-01-2025 10:03|
|   1005|      05-01-2025|  West|     Electronics|        120|De

In [10]:
df_filled = df.na.fill({"status": "Unknown"})

df_filled.show()

+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|user_id|transaction_date|region|product_category|sale_amount| city|age|subscription|price|store_id|             email|username|   status|   raw_timestamp|
+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|   1001|      01-01-2025|  West|     Electronics|        100|Delhi| 18|     Premium| NULL|    S101|              NULL|    NULL|  Unknown|01-01-2025 10:00|
|   1002|      02-01-2025|  West|       Furniture|        105|Delhi| 19|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|
|   1003|      03-01-2025|  West|        Clothing|        110|Delhi| 20|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|03-01-2025 10:02|
|   1004|      04-01-2025|  West|         Grocery|        115|De

Q6: Write a query to find the total count of records for each city in a DataFrame, but
only for cities where the count is greater than 100.

In [11]:
from pyspark.sql.functions import count

df.groupBy("city") \
  .agg(count("*").alias("record_count")) \
  .filter("record_count > 100") \
  .show()

+-------+------------+
|   city|record_count|
+-------+------------+
| Mumbai|         110|
|Kolkata|         110|
|  Delhi|         120|
+-------+------------+



Q7: How does the immutability of Spark DataFrames affect how you perform "data
cleaning" steps like dropping columns or renaming them?

In [12]:
#Dropping a Column
df_new = df.drop("status")

In [13]:
# Renaming a Column
df_renamed = df.withColumnRenamed(
    "sale_amount",
    "sales_amount"
)

Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and
30 (inclusive) and the subscription is 'Premium'.

In [14]:
from pyspark.sql.functions import col

df.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
).show()

+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|user_id|transaction_date|region|product_category|sale_amount| city|age|subscription|price|store_id|             email|username|   status|   raw_timestamp|
+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|   1001|      01-01-2025|  West|     Electronics|        100|Delhi| 18|     Premium| NULL|    S101|              NULL|    NULL|     NULL|01-01-2025 10:00|
|   1003|      03-01-2025|  West|        Clothing|        110|Delhi| 20|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|03-01-2025 10:02|
|   1005|      05-01-2025|  West|     Electronics|        120|Delhi| 22|     Premium|  120|    S101|user1005@gmail.com|user1005|Completed|05-01-2025 10:04|
|   1007|      07-01-2025|  West|        Clothing|        130|De

Q9: When cleaning a dataset, why is it often better to handle null values before
performing mathematical aggregations like sum() or avg()?

In [15]:
df_clean = df.na.fill({"price": 0})

In [16]:
from pyspark.sql.functions import avg

df_clean.agg(avg("price")).show()

+----------+
|avg(price)|
+----------+
|   342.575|
+----------+



Q10: Write the code to revise a column named raw_timestamp by casting it to a
TimestampType and renaming it to event_time.

In [17]:
from pyspark.sql.functions import col, to_timestamp

df_updated = (
    df.withColumn(
        "event_time",
        to_timestamp(col("raw_timestamp"), "dd-MM-yyyy HH:mm")
    )
    .drop("raw_timestamp")
)

df_updated.show()

+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount| city|age|subscription|price|store_id|             email|username|   status|         event_time|
+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+-------------------+
|   1001|      01-01-2025|  West|     Electronics|        100|Delhi| 18|     Premium| NULL|    S101|              NULL|    NULL|     NULL|2025-01-01 10:00:00|
|   1002|      02-01-2025|  West|       Furniture|        105|Delhi| 19|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|2025-01-02 10:01:00|
|   1003|      03-01-2025|  West|        Clothing|        110|Delhi| 20|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|2025-01-03 10:02:00|
|   1004|      04-01-2025|  West|         Groc

Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it
considered a wide transformation?

In [18]:
df.groupBy("store_id").sum("price")

DataFrame[store_id: string, sum(price): bigint]

In [19]:
from pyspark.sql.functions import sum

df.groupBy("store_id") \
  .agg(sum("price").alias("total_revenue")) \
  .show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|    S102|        39985|
|    S104|        12540|
|    S101|        44520|
|    S103|        39985|
+--------+-------------+



Q12: Write a code snippet that identifies and removes rows where the email column
contains null values OR the username is an empty string.

In [20]:
from pyspark.sql.functions import col

df_clean = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

df_clean.show()

+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|user_id|transaction_date|region|product_category|sale_amount| city|age|subscription|price|store_id|             email|username|   status|   raw_timestamp|
+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|   1002|      02-01-2025|  West|       Furniture|        105|Delhi| 19|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|
|   1003|      03-01-2025|  West|        Clothing|        110|Delhi| 20|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|03-01-2025 10:02|
|   1004|      04-01-2025|  West|         Grocery|        115|Delhi| 21|       Basic|  115|    S101|user1004@gmail.com|user1004|Completed|04-01-2025 10:03|
|   1005|      05-01-2025|  West|     Electronics|        120|De

Q13: How do you use the .agg() function to calculate multiple statistics at once, such as
the min, max, and mean of the price column?

In [21]:
from pyspark.sql.functions import min, max, avg

df.agg(
    min("price").alias("Minimum_Price"),
    max("price").alias("Maximum_Price"),
    avg("price").alias("Average_Price")
).show()

+-------------+-------------+------------------+
|Minimum_Price|Maximum_Price|     Average_Price|
+-------------+-------------+------------------+
|          105|          690|349.56632653061223|
+-------------+-------------+------------------+



Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when
your source data contains messy or inconsistent date formats?

In [23]:
df = spark.read.csv(
    "sales_dataset.csv",
    header=True,
    inferSchema=True
)


Q15: Write a final processing pipeline that:

• Filters out duplicates.  
• Fills null prices with 0.  
• Groups by store_id to calculate total revenue.



In [24]:
from pyspark.sql.functions import sum
final_result = (
    df
    .dropDuplicates()
    .na.fill({"price": 0})
    .groupBy("store_id")
    .agg(sum("price").alias("total_revenue"))
)
final_result.show()


+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|    S102|        39985|
|    S104|        12540|
|    S101|        44520|
|    S103|        39985|
+--------+-------------+

